<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP7.2C/MP7.2C_panel_strike.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP7.2C · Where Was the Panel Struck?

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Physics-Informed Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'problem.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.2-heat-and-wave/")
PROJECT = (['mp72c.py', 'MP7.2C_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP7.2C/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex07.2-heat-and-wave", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex07.2-heat-and-wave", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'problem', 'mp72c']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import problem as hs
import mp72c as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP7.2C_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

pb = hs                                                  # Ex07.2's problem.py, by its name in Ex07.2

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

A panel is struck and nobody saw where. It happens to the skin of an aircraft hit by a bird or hail, to a car's body panel, to a machine guard hit by a flying part, to a membrane roof under a falling branch. The damage, if any, is where the blow landed, and an inspection must know **where to look and how hard the blow was**. What is available is what a few sensors recorded: the displacement at three to six points of the panel, sampled every 0.1 ms for the 20 ms after the blow.

This is an **inverse problem**: Ex07.2's third notebook computed the motion of a panel from a known strike; here the motion is measured at a few points and the strike is unknown. And it is asked **again and again** - for every blow a monitoring system records - so what matters is not only the answer but the time to it. A least-squares fit with a wave solver takes about a minute per blow; this mini project builds one network that answers a new set of readings in a millisecond.

## A2 · The panel, and what is new against Ex07.2

The panel is Ex07.2's: 0.40 × 0.40 m, tensioned, clamped on its four edges, the wave travelling at 80 m/s. **New:** two things a real panel has.

* A **reinforcing strip** bonded across it, along $y$ at $x$ = 0.27 m. It doubles the mass per area on its centre line and tapers off over about 12 mm on either side. The tension is unchanged, so the wave is slower over the strip, $80/\sqrt2$ = 57 m/s on its centre line, and part of each wave is reflected there.
* A **bolt** through a hole at (0.10, 0.20) m carrying a small bracket: the bolt, its nut and washers and the bracket's foot add ten times the panel's own mass per area over a disc of 15 mm radius. Under it the wave crawls at 24 m/s.

The strike is a hammer blow: at $t = 0$ the panel is flat and a patch of it moves, $v\,e^{-r^2/2s^2}$ with $r$ the distance from the strike $(x_0, y_0)$ and $s$ = 25 mm the hammer's footprint, which is known. **Hidden:** $x_0$, $y_0$ and the peak velocity $v$.

| | value |
|---|---|
| the panel | 0.40 × 0.40 m, clamped edges, wave speed 80 m/s away from the strip and the bolt (Ex07.2) |
| the strip | along $y$ at $x$ = 0.27 m; mass per area × 2 on its centre line, a Gaussian profile of 12 mm half-width (assumed) |
| the bolt | at (0.10, 0.20) m; mass per area × 11 over a disc of 15 mm radius (assumed) |
| the strike | anywhere with $x_0$, $y_0$ from 0.06 to 0.34 m and at least 40 mm from the bolt; $v$ = 0.5 to 1.5 m/s; footprint $s$ = 25 mm (assumed) |
| the sensors | six positions, at most six read; the displacement every 0.1 ms for 20 ms, 201 readings each |
| the noise | 2, 5 or 10 µm, one standard deviation, independent at every reading (assumed for displacement sensors) |

**Why no exact solution exists.** Ex07.2's panel is uniform, and any motion of it is a sum of its modes $\sin(m\pi x/L)\sin(n\pi y/L)$, each ringing at its own frequency. With a mass that depends on the place, these are no longer the panel's modes, the equation does not separate, and only a numerical solution remains. **The strength of the change** is a factor $S$ on both the strip's and the bolt's extra mass: at $S = 0$ the panel is Ex07.2's, and the solver must reproduce Ex07.2's exact sum of modes. Part C checks it.

In [ ]:
# The panel, the strip, the bolt, the sensors, and test case 1's readings
G = np.load("MP7.2C_truth.npz")
fig, ax = plt.subplots(1, 2, figsize=(13, 5.2), gridspec_kw={"width_ratios": [1, 1.6]})
mp.draw_panel(ax[0], strike=G["case1_strike"])            # the hidden strike of case 1, shown here only
ax[0].legend(fontsize=8, loc="upper right")
t = G["t"] * 1e3
for j in range(6):
    ax[1].plot(t, G["case1_readings"][:, j] * 1e3, lw=0.9, label=f"sensor {j + 1}")
ax[1].set_xlabel("t [ms]"); ax[1].set_ylabel("displacement [mm]"); ax[1].legend(fontsize=8, ncol=2)
ax[1].set_title("case 1: what the six sensors record, noise included", fontsize=10)
plt.tight_layout(); plt.show()
ex = mp.modal_sum(G["case1_strike"], G["sensors"][:, 0], G["sensors"][:, 1], G["t"])   # the same strike on Ex07.2's uniform panel
print("sensors [mm]:", (G["sensors"] * 1e3).round().astype(int).tolist())
print(f"case 1: largest reading {np.abs(G['case1_clean']).max()*1e3:.3f} mm; the same strike on Ex07.2's uniform panel "
      f"(exact sum of modes) differs by up to {np.abs(G['case1_clean'] - ex).max()*1e3:.3f} mm")

**What you should see.** Left, the panel: the strip and the bolt's disc in grey, the six sensors in red, and case 1's strike, which you are shown here only - the readings are what you work from. Right, the six records of case 1: the two sensors nearest the strike, 4 and 5, move at once - the hammer's footprint reaches them - and the others stay still until the wave arrives, 2 to 3 ms after the blow; then all swing with amplitudes of up to 0.14 mm as the waves reflect from the edges, the strip and the bolt. On Ex07.2's uniform panel the same strike gives readings that differ by up to 0.18 mm, more than the readings themselves: the strip and the bolt change the motion completely, and the exact sum of modes no longer describes it.

## A3 · The equation and its conditions

With $m(x, y)$ the mass per area relative to Ex07.2's - 1 on the bare panel, up to 2 on the strip and 11 under the bolt:

$$m(x, y)\,\frac{\partial^2 u}{\partial t^2} = c^2\left(\frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2}\right) \quad\text{on the panel}, \qquad c = 80\ \mathrm{m/s}$$

| where | condition | why it holds |
|---|---|---|
| the four edges | $u = 0$ | the panel is clamped in its frame (Ex07.2) |
| the start, $t = 0$ | $u = 0$ | the panel is flat when the hammer lands |
| the start, $t = 0$ | $\partial u/\partial t = v\,e^{-r^2/2s^2}$ | the blow gives a patch of the panel its velocity; the second initial condition of Ex07.2 |
| the sensors | $u(x_j, y_j, t_k)$ = reading ± noise | the measurements: at three to six points, every 0.1 ms |

The equation is **hyperbolic** (Ex07.2): disturbances travel at the wave speed and are not damped, so the strike's position is written in the **arrival times** of its wave at the sensors, and its strength in their amplitudes.

## A4 · Why the inverse problem is hard

* **Few points, many reflections.** Three to six sensors; after the first arrival every record is a superposition of reflections from the edges, the strip and the bolt.
* **A misfit with many valleys.** Shift a guessed strike by half a wavelength and some peaks of the predicted records line up with the wrong peaks of the measured ones: a least-squares fit started far from the answer can end in a wrong valley. Part C's fit therefore starts from nine places.
* **The noise** is 1.5 to 10 % of the largest reading, and in case 4 only three sensors are read.
* **The motion of the whole panel** is wanted too, and the readings see it only at three to six points: it follows from the strike through the equation, so it is only as good as the strike found.

## A5 · What is asked

Four test cases of rising difficulty, all public in `MP7.2C_truth.npz` - the readings to work from, and the hidden answers to score against afterwards:

| case | sensors read | noise [µm] |
|---|---|---|
| 1 | all six | 2 |
| 2 | 1, 3, 4, 6 (the four nearest the corners) | 5 |
| 3 | 1, 2, 5 | 5 |
| 4 | 2, 3, 5 | 10 |

| quantity | required, in every case |
|---|---|
| the strike's position $(x_0, y_0)$ | within **10 mm** |
| the strike's peak velocity $v$ | within **10 %** |
| the motion of the panel that the answer implies (part C's solver run with the strike found, at the nodes every 8 mm, every 0.5 ms) | within **5 %** of the true motion's peak |
| the time to answer a new set of readings | at least **10 times shorter** than a new least-squares fit with the reference model (part C) |

One network, trained once over the whole range of strikes, answers a new set of readings by being evaluated: **amortised** inference. Its training, the simulation of its training data included, is reported apart, as the offline cost.

**The agreed accuracy.** The proposal's targets are kept, and part C shows that the readings can support them: the information they carry (the Cramér-Rao bound) leaves a standard deviation of 0.11 to 1.30 mm on the position, 0.13 to 1.34 % on the velocity and 0.18 to 2.01 % of the peak on the motion, and the least-squares reference meets all three in every case. **The motion is the binding requirement**: 5 % of the peak is about 2.5 standard deviations in case 4, and a position about 2 mm off already costs a few per cent of the motion, while 10 mm and 10 % are far from the bound.

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

An amortised network inverts the readings directly:

$$\text{readings } (201 \times 6) \text{ and which sensors are read } \longmapsto\ (x_0,\ y_0,\ v)$$

Trained on many strikes whose answer is known, it learns the map from readings to strike once. A new set of readings then costs one evaluation, against the hundreds of wave simulations a least-squares fit needs.

Its physics comes from the **forward model**: every training example is the panel's motion computed by part C's finite differences for a random strike, read at the six sensors. Noise is added afresh at every step of training, and a random subset of the sensors is removed, so one network learns to read through the noise and serves any three to six sensors. A **physics-informed** alternative - an inverse PINN for $u(x, y, t)$ and the strike, fitted to one set of readings and the wave equation, as the first proposal had it - answers one set of readings at a time and must be trained again for each; you may compare it in question 8.

The scales of the problem:

In [ ]:
# The scales: the waves, and how far the readings move when the strike moves
print(f"wave speed: {mp.C:.0f} m/s on the bare panel, {mp.C/np.sqrt(2):.0f} m/s on the strip's line, "
      f"{mp.C/np.sqrt(1 + mp.BOLT_BETA):.0f} m/s under the bolt")
print(f"time for a wave to cross the panel: {mp.L/mp.C*1e3:.1f} ms; the record is {mp.T_END*1e3:.0f} ms, so it holds about "
      f"{mp.T_END*mp.C/mp.L:.0f} crossings")
print(f"the hammer's footprint s = {mp.HAMMER*1e3:.0f} mm: the wave it sends is about {2*np.pi*mp.HAMMER*1e3:.0f} mm long, "
      f"{2*np.pi*mp.HAMMER/mp.C*1e3:.1f} ms at a sensor")
p = mp.Panel(mp.REF_H)
q = np.array(G["case1_strike"])
r0 = p.run([q])[0]
for dq, name in ((np.array([0.005, 0, 0]), "x0 + 5 mm"), (np.array([0, 0.005, 0]), "y0 + 5 mm"), (np.array([0, 0, 0.05]), "v + 5 %")):
    print(f"case 1's strike moved {name}: the readings change by up to {np.abs(p.run([q + dq])[0] - r0).max()*1e6:.1f} um "
          f"(the noise of case 1: {float(G['case1_noise'])*1e6:.0f} um)")

**What you should see.** The wave crosses the 0.40 m panel in 5 ms, so a 20 ms record holds four crossings: plenty of reflections. The hammer's 25 mm footprint sends a wave about 157 mm long, 2 ms at a sensor. Moving case 1's strike by 5 mm changes some reading by 12 to 15 µm, six to eight times the noise of case 1, and a 5 % faster blow by 6.8 µm: the readings fix the position to a fraction of a millimetre (part C), and the velocity to a fraction of a per cent.

## B2 · The physics the training data obey

| what | how it enters |
|---|---|
| the wave equation with the strip and the bolt | each training example is one solution of it, computed by part C's finite differences on the 2 mm grid |
| the clamped edges and the flat start | in the forward model |
| the strike's footprint | known: a Gaussian of 25 mm; the network finds $x_0$, $y_0$ and $v$ |
| the noise | added at every training step, its standard deviation drawn from 2 to 10 µm |
| missing sensors | at every step a random three to six of the six are kept; the others are set to zero and flagged |

**Reciprocity makes the training data cheap.** The panel is linear, and its equation is symmetric once the mass is taken as the weight. So the reading of sensor $j$ at $t$ for a strike of initial velocity $v_0(x, y)$ is

$$u(x_j, y_j, t) = \sum_\text{nodes} G_j(\text{node}, t)\; m(\text{node})\; v_0(\text{node})$$

where $G_j$ is the motion that an impulse of velocity $1/m$ at sensor $j$ makes - the same finite-difference march, run backwards from the sensor. Six marches (one per sensor) give $G$; after that the readings of any strike are one weighted sum, exactly the forward model's. `mp.Readings(h)` builds it; part C checks that it reproduces the forward model.

## B3 · The training data

`mp.sample_strikes(n, seed)` draws strikes uniformly over the range (at least 40 mm from the bolt); `mp.simulate(strikes, operator=OP)` returns their clean readings (n, 201, 6) [m] from a `mp.Readings(0.002)` operator built once. On one CPU core the six marches take about 5 s and then about 15 ms per strike: 40 000 strikes in about 10 minutes, a cost paid once, offline, like the training (on a GPU runtime a few times less). The cell below makes a few to show them; part D makes the training set.

In [ ]:
# A few training examples: strikes drawn over the range, and their readings
t0 = time.perf_counter()
OP = mp.Readings(0.002)                                   # six marches on the 2 mm grid: the readings of any strike
print(f"reciprocity operator: {time.perf_counter() - t0:.1f} s, {OP.G.nbytes/1e6:.0f} MB")
Qd = mp.sample_strikes(300, seed=1)
t0 = time.perf_counter()
Rd = mp.simulate(Qd, operator=OP)
print(f"{len(Qd)} strikes in {time.perf_counter() - t0:.1f} s")
check = mp.Panel(0.002).run(Qd[:3])                       # the forward model, marched, for three of them
print(f"reciprocity against the march: largest difference {np.abs(Rd[:3] - check).max() / np.abs(check).max():.1e} of the largest reading")
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
mp.draw_panel(ax[0], label_sensors=False)
ax[0].scatter(Qd[:, 0] * 1e3, Qd[:, 1] * 1e3, c=Qd[:, 2], s=10, cmap="viridis", zorder=4)
ax[0].set_title("300 strikes, coloured by their velocity", fontsize=10)
ax[1].scatter(np.hypot(Qd[:, 0] - 0.2, Qd[:, 1] - 0.2) * 1e3, np.abs(Rd).max((1, 2)) * 1e3, s=6, c=Qd[:, 2], cmap="viridis")
ax[1].set_xlabel("distance of the strike from the panel's centre [mm]"); ax[1].set_ylabel("largest reading [mm]")
plt.tight_layout(); plt.show()

**What you should see.** The operator takes a few seconds and holds 195 MB. 300 strikes come out in a few seconds; the readings by reciprocity and by marching the forward model agree to $2\times10^{-7}$ of the largest reading - the rounding of single precision. Left, the strikes over the panel, none near the bolt; right, the largest reading of each: from about 0.06 to 0.3 mm, larger for a faster blow. It tends to grow with the distance from the panel's centre because the sensors sit towards the corners: a strike near one of them gives a large reading there.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Three solvers, three jobs

| | the **forward model** | the **reference model** | the **ground truth** |
|---|---|---|---|
| what it is | finite differences in space, the leapfrog in time | least squares over $(x_0, y_0, v)$, the forward model on the engineering grid inside | the forward model on two fine grids, extrapolated |
| grid | any | 4 mm (100 × 100 cells) | 2 mm and 1 mm, combined by Richardson extrapolation |
| its job | to make the training data (on the 2 mm grid, by reciprocity) | to be **timed** against the network: the online cost of a new set of readings | to make the test cases' readings and motions and to **score** the answers |

## C2 · The forward model, step by step

1. **Nodes.** The panel is covered by nodes $h$ apart, the edges included; $u = 0$ on the edge nodes.
2. **The mass.** Each node carries its $m(x, y)$: the strip's Gaussian and the bolt's disc, a smooth plateau whose edge falls over about 3 mm, so that the coefficient is smooth and the scheme keeps its order.
3. **Space: the five-point Laplacian**, Ex07.2's, divided by the node's mass.
4. **Time: the leapfrog**, $u^{n+1} = 2u^n - u^{n-1} + (c\,\Delta t/h)^2\,\nabla_h^2 u^n / m$, Ex07.2's scheme. It is stable while $c\sqrt2\,\Delta t/h < 1$; the step is chosen so that $c\sqrt2\,\Delta t/h = 0.71$ on every grid and a whole number of steps fills the sensors' 0.1 ms.
5. **The start.** $u^0 = 0$ and $u^1 = \Delta t\,v_0 + \tfrac{\Delta t^3}{6}\,c^2\nabla_h^2 v_0/m$: Taylor's series with both initial conditions, accurate enough that the scheme stays second order.
6. **Readings** are the values at the sensors' nodes every 0.1 ms; the **motion** the values at the nodes every 8 mm every 0.5 ms.

`mp.Panel(h).run(strikes)` marches any number of strikes at once.

## C3 · The reference model: least squares

`mp.least_squares_fit(readings, sensors, noise)` finds the strike whose predicted readings are closest to the measured ones, each residual divided by the noise, with `scipy.optimize.least_squares` (trust-region reflective, within the range's bounds, derivatives by finite differences), the forward model on the 4 mm grid. Because the misfit has many valleys (part A), it starts from **nine** strikes on a 3 × 3 grid over the panel and keeps the best fit: some 600 forward runs per set of readings.

In [ ]:
# The ground truth: convergence on case 1's strike, and the zero-strength check, from the truth file
G = np.load("MP7.2C_truth.npz")
print("grid [mm]   largest reading [mm]   change from the grid before [% of it]   march [s]")
for h, p_, c, s in zip(G["conv_h_mm"], G["conv_peak"], G["conv_change"], G["conv_seconds"]):
    print(f"   {h:3.0f}           {p_*1e3:.4f}                {c*100:7.3f}                         {s:5.1f}")
c = G["conv_change"][1:]
print("ratios of the changes:", np.round(c[:-1] / c[1:], 2))
print(f"S = 0 (no strip, no bolt) against Ex07.2's exact sum of modes: {float(G['zero_strength_fine'])*100:.3f} % of the largest "
      f"reading on the 1 mm grid, {float(G['zero_strength_extrapolated'])*100:.4f} % extrapolated")

**What you should see.** Four grids, each twice as fine in space and in time. The readings change by 10.5, 2.8 and 0.71 % of the largest reading: each change 3.7 and then 3.9 times smaller than the one before, the **second order** of the scheme. Richardson extrapolation from the 2 mm and 1 mm grids, $(4u_\mathrm{fine} - u_\mathrm{coarse})/3$, moves the readings by 0.24 % and leaves the ground truth within a few hundredths of a per cent: a hundred times finer than the 5 % asked of the motion. At $S = 0$ the same scheme gives Ex07.2's exact sum of modes to 0.15 % on the 1 mm grid and to 0.007 % extrapolated: the code is right where the answer is known. The 4 mm grid of the reference model is 2.8 % off on its own - about the noise of the easier cases - which is why it is an engineering grid, not the ground truth.

In [ ]:
# The four test cases: the least-squares reference on each, and the motion its answer implies
CASES = [str(c) for c in G["cases"]]
panel = mp.Panel(mp.REF_H)                                 # the engineering grid, built once
LSQ = {}
print("case  sensors  noise [um] | position error [mm]  velocity error [%]  motion error [% of peak] | forward runs   time [s]")
for n in CASES:
    m = G[f"case{n}_mask"].astype(bool)                     # which sensors were read
    noise = float(G[f"case{n}_noise"])
    f = mp.least_squares_fit(G[f"case{n}_readings"][:, m], G["sensors"][m], noise, panel=panel)
    s = G[f"case{n}_strike"]
    e_p = np.hypot(*(f["strike"][:2] - s[:2])) * 1e3
    e_v = (f["strike"][2] / s[2] - 1) * 100
    e_m = mp.motion_error(f["strike"], G[f"case{n}_motion"]) * 100
    LSQ[n] = dict(f, position_mm=e_p, velocity=e_v, motion=e_m)
    print(f"  {n}      {m.sum()}       {noise*1e6:4.0f}    |       {e_p:5.2f}               {e_v:+6.2f}                {e_m:5.2f}"
          f"           |     {f['runs']:4d}       {f['seconds']:5.1f}")
t_lsq = float(np.median([v["seconds"] for v in LSQ.values()]))
REF = {"t_ref_s": t_lsq, "lsq": LSQ}                         # kept for part E
print(f"\nleast squares: {t_lsq:.1f} s per new set of readings (the median of the four); "
      f"the network must answer in under {t_lsq*1e3/10:.0f} ms")

**What you should see.** The least-squares reference finds the strike within 0.10 to 0.90 mm, its velocity within 0.42 % in cases 1 to 3 and 2.18 % in case 4, and the motion its answer implies within 0.26 to 3.03 % of the true motion's peak: all three requirements are met in every case, the hardest being case 4, three sensors at 10 µm. Each fit takes 545 to 657 forward runs on the 4 mm grid, about a minute on one CPU core (30 to 61 s in our runs; this runtime's median is kept in `REF`). The motion error is computed by running the ground truth's solver with the strike found: about 9 s per strike, which is why it is not part of the online time.

In [ ]:
# What the readings can tell: the Cramér-Rao bound at each case's true strike
print("case | smallest possible standard deviation of: position [mm]  velocity [%]  motion [% of peak]")
for n in CASES:
    m = G[f"case{n}_mask"].astype(bool)
    b = mp.information_bound(G[f"case{n}_strike"], G["sensors"][m], float(G[f"case{n}_noise"]))
    print(f"  {n}  |                                       {b[0]:5.2f}          {b[1]:5.2f}          {b[2]:5.2f}")

**What you should see.** The **Cramér-Rao bound**: from how fast the readings change as each parameter of the strike changes (the Jacobian $J$, by central differences on the forward model) and the noise $\sigma$, the covariance $\sigma^2 (J^\top J)^{-1}$ is the smallest any unbiased method can reach. It is a statement about the readings, not about a method. One standard deviation: the position 0.11 to 1.30 mm, the velocity 0.13 to 1.34 %, the motion 0.18 to 2.01 % of its peak - the largest in case 4, with its three sensors and the largest noise. The required 10 mm and 10 % are far above the bound; the 5 % on the motion is 2.5 standard deviations in case 4, met there by least squares with 3.0 %. A network trained over the range of strikes can come close to the bound, but no method can promise much better than it.

---

# Part D · The Physics-Informed Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** (TODO 2, including how many strikes you simulate), and the **figures** (TODO 3).

**The contract.** Your network is a PyTorch module called `net`, used as `net(readings, mask)`:

| argument | meaning | shape |
|---|---|---|
| `readings` | the displacement [m] at the six sensors every 0.1 ms from 0 to 20 ms; zero where a sensor is not read | `(n, 201, 6)` |
| `mask` | 1 for a sensor that was read, 0 for one that was not | `(n, 6)` |
| returns | $x_0$, $y_0$ [m] and $v$ [m/s] of the strike | `(n, 3)` |

Whatever you compute inside - features of the readings, scaling, several heads, the precision of the arithmetic - stays inside `net`.

## D2 · The training data and the loss, given

`make_data(n, seed)` draws n strikes and simulates their clean readings with part B's reciprocity operator; `noisy(readings, sigma, sensors, generator)` adds fresh noise of a standard deviation drawn per example from `sigma` [m] and keeps a random number of the six sensors between `sensors[0]` and `sensors[1]`. `loss(prediction, strikes)` is the mean squared error of the position in units of 10 mm and of the logarithm of the velocity in units of 0.1 (10 %): both errors count alike at the tolerance.

In [ ]:
# The training data and the loss, given ---------------------------------------------
OP = mp.Readings(0.002)                                    # the reciprocity operator of part B, built once (a few seconds)

def make_data(n, seed):
    """n strikes over the range and their clean readings: strikes (n, 3), readings (n, 201, 6) [m], as tensors."""
    Q = mp.sample_strikes(n, seed)
    R = mp.simulate(Q, operator=OP)
    return torch.tensor(Q, device=DEVICE), torch.tensor(R, device=DEVICE)

def noisy(readings, sigma=(2e-6, 10e-6), sensors=(3, 6), generator=None):
    """The readings with fresh noise (a σ drawn per example from sigma [m]) and a random subset of the sensors kept
    (between sensors[0] and sensors[1] of the six); the others zeroed and flagged. Returns readings, mask."""
    n = readings.shape[0]
    s = sigma[0] + (sigma[1] - sigma[0]) * torch.rand(n, 1, 1, generator=generator, device=DEVICE)
    r = readings + s * torch.randn(readings.shape, generator=generator, device=DEVICE)
    keep = torch.randint(sensors[0], sensors[1] + 1, (n, 1), generator=generator, device=DEVICE)   # how many are read
    rank = torch.argsort(torch.rand(n, 6, generator=generator, device=DEVICE), 1).argsort(1)       # a random order
    mask = (rank < keep).to(r.dtype)
    return r * mask[:, None, :], mask

def loss(prediction, strikes):
    """Mean squared error of x0 and y0 in units of 10 mm and of ln v in units of 0.1."""
    e_pos = (prediction[:, :2] - strikes[:, :2]) / 0.01
    e_v = torch.log(prediction[:, 2:] / strikes[:, 2:]) / 0.1
    return (torch.cat([e_pos, e_v], 1) ** 2).mean()

## D3 · TODO 1 · Your network

Write the class `Net` and make `net = Net().to(DEVICE)`. Decide:

* **what the network sees.** The readings are proportional to the strike's velocity, 0.03 to 0.3 mm, and their shape carries the position; a missing sensor must be told apart from a still one (the mask). Whether every 0.1 ms sample is needed: the wave a 25 mm hammer sends lasts about 2 ms at a sensor;
* **what it returns, and how.** Scaled outputs; the velocity as a positive number;
* **its size.** Layers, neurons, activation; single or double precision.

Remember part A: the position is written in the arrival times and the velocity in the amplitudes.

In [ ]:
# TODO 1 - your network. net(readings (n, 201, 6) [m], mask (n, 6)) -> (n, 3): x0, y0 [m], v [m/s]
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ...                                              # your layers

    def forward(self, readings, mask):
        ...                                              # return (n, 3)

torch.manual_seed(72)
net = Net().to(DEVICE)
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training

Simulate your training set with `make_data` (decide how many strikes: about 15 ms each on one CPU core, and the simulation is part of the offline cost), then train `net` with `noisy` and `loss`. Decide the optimiser, the epochs, the batch size, the learning rate and its schedule, the noise and the sensors you train with, and whether to keep strikes apart for validation. Keep the loss history for TODO 3, and **measure the offline time**, simulation included: set `TRAIN_SECONDS`.

`quick_score()` below gives, over the four test cases, the worst position error [mm] and the worst velocity error [%] (the motion error is computed in part E: it takes a simulation per answer).

In [ ]:
# A quick score, given: the worst errors over the four test cases
G = np.load("MP7.2C_truth.npz")
CASES = [str(c) for c in G["cases"]]
CASE_R = torch.tensor(np.stack([G[f"case{n}_readings"] for n in CASES]), device=DEVICE)   # (4, 201, 6) [m]
CASE_M = torch.tensor(np.stack([G[f"case{n}_mask"] for n in CASES]), device=DEVICE)       # (4, 6)

def answer(readings=CASE_R, mask=CASE_M):
    """The network's answer to sets of readings: (n, 3) as a NumPy array, x0, y0 [m], v [m/s]."""
    with torch.no_grad():
        return net(readings, mask).double().cpu().numpy()

def case_errors(p):
    """Per case: the position error [mm] and the velocity error [%]."""
    out = []
    for i, n in enumerate(CASES):
        s = G[f"case{n}_strike"]
        out.append((np.hypot(p[i, 0] - s[0], p[i, 1] - s[1]) * 1e3, (p[i, 2] / s[2] - 1) * 100))
    return np.array(out)

def quick_score():
    e = case_errors(answer())
    return np.abs(e[:, 0]).max(), np.abs(e[:, 1]).max()

In [ ]:
# TODO 2 - your training
history = []
t0 = time.perf_counter()
...                                                      # simulate your training set, then train; append the loss to history
TRAIN_SECONDS = time.perf_counter() - t0
print(f"offline {TRAIN_SECONDS:.0f} s; worst errors: position %.2f mm, velocity %.2f %%" % quick_score())

## D5 · TODO 3 · Your figures

Draw what you need to understand your network and to answer the report: at least the loss during training, and the network's answers against the truth on held-out strikes (position error over the panel, velocity). Save each figure you want in the report as `MP7.2C_report_<name>.png` (in `cc.output_path`).

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores and keeps the answers in `R`: its answers to the four test cases and the motion each answer implies (the ground truth's solver run with the strike found, about 9 s each); its answers to 100 fresh noisy sets of readings of case 4 (to see the spread the noise causes); and the **online time** for 1, 10 and 100 new sets of readings, against a new least-squares fit (part C's time, measured on this runtime).

In [ ]:
# The evaluation, given ------------------------------------------------------------
net.eval()
out = {"cases": np.array(CASES), "answer": answer()}                    # (4, 3)
out["motion_error"] = np.array([mp.motion_error(out["answer"][i], G[f"case{n}_motion"]) for i, n in enumerate(CASES)])
# 100 new sets of readings of case 4: the same strike, fresh noise of its σ, its three sensors
rng = np.random.default_rng(100)
clean, mask4 = G["case4_clean"], G["case4_mask"]
rep = (clean[None] + float(G["case4_noise"]) * rng.standard_normal((100,) + clean.shape)) * mask4
out["case4_repeat"] = answer(torch.tensor(rep, device=DEVICE), torch.tensor(np.tile(mask4, (100, 1)), device=DEVICE))
timing = {}
for k in (1, 10, 100):
    r = torch.tensor(rep[:k], device=DEVICE); m = torch.tensor(np.tile(mask4, (k, 1)), device=DEVICE)
    timing[f"net_{k}"] = middle(lambda: answer(r, m))
out.update({k: np.array(v) for k, v in timing.items()})
out["lsq_seconds"] = np.array(REF["t_ref_s"])
out["train_seconds"] = np.array(TRAIN_SECONDS)
out["n_weights"] = np.array(sum(p.numel() for p in net.parameters()))
out["device"] = np.array(DEVICE)
R = out                                                  # part E reads the answers from here
torch.save(net.state_dict(), cc.output_path("MP7.2C_net.pt"))
print(f"online, one new set of readings: network {timing['net_1']*1e3:.2f} ms, least squares {REF['t_ref_s']*1e3:.0f} ms "
      f"({REF['t_ref_s']/timing['net_1']:.0f} times)")
print("motion error of each case's answer [% of peak]:", np.round(out["motion_error"] * 100, 2))

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the hidden values in the truth file `G` and the least-squares reference `REF` of part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the eight questions.

In [ ]:
# What part E uses, all from the parts above
G = np.load("MP7.2C_truth.npz")
print("in R:", ", ".join(R))
print(f"least squares (part C): {REF['t_ref_s']:.1f} s per new set of readings")

## E2 · TODO 4 · The goal metrics

Compute, for each of the four cases, and put in `METRICS` (a name and a number each):

* the position error [mm], the velocity error [%] and the motion error [% of peak], the network's and the least-squares reference's (`REF["lsq"]`);
* the network's online time for one set of readings, the least-squares time, and their ratio;
* the offline time.

Then say in one line each whether the four requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>50}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP7.2C_report_<name>.png`, the figures your answers refer to. At least: each case's true strike, the network's and the least-squares' on the panel; the spread of the network's answers to the 100 noisy repeats of case 4 (`R["case4_repeat"]`); the motion of the panel at one instant, true and implied by your answer, for the hardest case; and the online times for 1, 10 and 100 sets of readings against least squares.

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the eight questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 4 are about your network and its result; 5 to 7 ask you to use it; 8 concludes. The next cell writes `MP7.2C_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: its inputs (and any features computed from the readings), layers, neurons, outputs, precision and number of weights. Why this design?": """
""",
    "2. Your training data and training: how many strikes you simulated and how long that took, the noise and the sensors, the optimiser, epochs and the offline time in seconds.": """
""",
    "3. The four test cases: give the network's position error [mm], velocity error [%] and motion error [% of peak] for each. Which requirements are met, and how do they compare with the least-squares reference's?": """
""",
    "4. Case 4 repeated 100 times with fresh noise: how large is the spread of the network's position and velocity? Compare it with the Cramér-Rao bound of part C.": """
""",
    "5. The online time for one new set of readings: the network against a new least-squares fit, in milliseconds, and the ratio. Then for 10 and 100 sets: how do the two grow?": """
""",
    "6. Where on the panel does your network find the strike least well (use the held-out strikes): near an edge, near the strip, near the bolt, far from the sensors? Explain from the waves.": """
""",
    "7. Which case is hardest, and why: the sensors read, their positions, the noise? How does your error grow from six to three sensors on the held-out strikes?": """
""",
    "8. To conclude: a monitoring system records 1 000 blows a year on this panel. Which method would you give it, given your measured accuracy, online time and offline time? What would it take to answer a panel with another strip, or an inverse PINN instead?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" for k, v in METRICS.items()]
    out = ["# MP7.2C — Where Was the Panel Struck?", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP7.2C_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP7.2C_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP7.2C_report.md into MP7.2C_report.pdf, with any figure
# saved as MP7.2C_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP7.2C_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP7.2C_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP7.2C_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP7.2C_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass